# BUSI replication — training lane

Set `VARIANT` and `CONFIG` in cell 3, then Runtime > Run all.

**If the runtime dies, just Run all again.** The script detects the mirrored
`last.pt`, resumes from it, and exits immediately if the run already finished.
You do not need to think about it.

Claim your job in the shared manifest sheet before you start so two lanes don't
duplicate work.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive; drive.mount('/content/drive')

In [ ]:
# Pin the version. YOLOv10 support has moved between the THU-MIG repo and
# Ultralytics more than once; do not let pip resolve this freely.
!pip -q install 'ultralytics==8.3.40' 'transformers>=4.44' --no-warn-conflicts
import ultralytics, torch; print(ultralytics.__version__, torch.__version__)

%cd /content
![ -d busi-repro ] || git clone -q https://github.com/YOUR_USER/busi-repro.git
%cd /content/busi-repro
!git pull -q

In [ ]:
VARIANT = 'yolov10b'    # <-- your assigned job
CONFIG  = 'B'           # A = stock hparams, B = Table 2 hparams
SEED    = 42

MIRROR = '/content/drive/MyDrive/busi-repro/results'
DATA_ZIP_URL = 'https://github.com/YOUR_USER/busi-repro/releases/download/v1/busi_yolo.zip'
EXPECTED_MD5 = 'PASTE_FROM_BUILD_STEP'

In [ ]:
# Fetch the ONE canonical dataset zip. Never rebuild it per-lane.
import hashlib, os, pathlib
if not pathlib.Path('/content/busi_yolo/data.yaml').exists():
    !wget -q -O /content/busi_yolo.zip {DATA_ZIP_URL}
    got = hashlib.md5(open('/content/busi_yolo.zip','rb').read()).hexdigest()
    assert got == EXPECTED_MD5, f'DATASET MISMATCH {got} != {EXPECTED_MD5}'
    !unzip -q -o /content/busi_yolo.zip -d /content
!ls /content/busi_yolo/train/images | wc -l
!ls /content/busi_yolo/test/images | wc -l

In [ ]:
# 5-epoch smoke test. Run this ONCE on your first session, never again.
# Catches a broken YOLOv10 load in 3 minutes instead of at hour 20.
SMOKE = False
if SMOKE:
    !python -m src.train --variant {VARIANT} --config {CONFIG} --epochs 5 \
        --mirror /tmp/smoke --project /tmp/smokeruns

In [ ]:
!python -m src.train --variant {VARIANT} --config {CONFIG} --seed {SEED} \
    --data /content/busi_yolo/data.yaml --project /content/runs --mirror {MIRROR}

In [ ]:
# preds.json for configs A/B. Cheap; always do it in the same session as training.
TAG = f'{VARIANT}_{CONFIG}' + ('' if SEED == 42 else f'_s{SEED}')
!python -m src.predict --run {MIRROR}/{TAG} --data /content/busi_yolo
!python -m src.metrics --preds {MIRROR}/{TAG}/preds.json --mode yolo

In [ ]:
# Fused predictions for configs C/D. Needs the MedSAM cache from the CPU lane.
CACHE = '/content/drive/MyDrive/busi-repro/medsam_cache'
!python -m src.predict --run {MIRROR}/{TAG} --data /content/busi_yolo --medsam-cache {CACHE}
!python -m src.metrics --preds {MIRROR}/{TAG}/preds_fused.json --mode fusion